# Lab 4: GraphRAG - Xây dựng hệ thống RAG dựa trên đồ thị tri thức

> **Mục tiêu:** Xây dựng và so sánh Flat RAG (TF-IDF) vs GraphRAG (Knowledge Graph + 2-hop traversal)  
> **Công cụ:** NetworkX, Matplotlib, Python  
> **Deliverables:** Source code, Knowledge Graph screenshot, Benchmark table 20 câu hỏi, Cost analysis

---

## Phần 1: Cài đặt thư viện (Environment Setup)

In [ ]:
# Cài đặt các thư viện cơ bản
!pip install networkx matplotlib pandas
# Cài đặt NodeRAG framework
# !pip install noderag
# Nếu sử dụng LangChain:
# !pip install langchain langchain-openai
print('✅ Thư viện đã sẵn sàng')

## Phần 2: Tech Company Corpus

Dữ liệu đầu vào gồm 40 câu mô tả về các công ty công nghệ lớn.

In [ ]:
from corpus import TECH_CORPUS

print(f'📚 Tổng số tài liệu trong corpus: {len(TECH_CORPUS)}')
print('\n=== 5 tài liệu mẫu ===')
for i, doc in enumerate(TECH_CORPUS[:5], 1):
    print(f'{i}. {doc}')

## Phần 3: Bước 1 - Trích xuất thực thể và quan hệ (Indexing)

Sử dụng LLM để đọc corpus và chuyển thành bộ ba **(Subject, Relation, Object)**.

**Ví dụ:**
- Input: `"OpenAI được thành lập bởi Sam Altman và Elon Musk vào năm 2015."`
- Output Triples:
  - `(OpenAI, FOUNDED_BY, Sam Altman)`
  - `(OpenAI, FOUNDED_BY, Elon Musk)`
  - `(OpenAI, FOUNDED_IN, 2015)`

In [ ]:
from graph_builder import PREDEFINED_TRIPLES

print(f'🔗 Tổng số triples được trích xuất: {len(PREDEFINED_TRIPLES)}')
print('\n=== Mẫu triples ===')
for triple in PREDEFINED_TRIPLES[:10]:
    print(f'  ({triple[0]}, {triple[1]}, {triple[2]})')
print('  ...')

## Phần 4: Bước 2 - Xây dựng Knowledge Graph (NetworkX)

**Lựa chọn A: NetworkX** - phù hợp chạy offline trong Notebook.

In [ ]:
from graph_builder import build_graph

# Xây dựng Knowledge Graph từ triples
G = build_graph()

print(f'📊 Số nodes: {G.number_of_nodes()}')
print(f'📊 Số edges: {G.number_of_edges()}')
print('\n=== Danh sách nodes ===')
for node, data in list(G.nodes(data=True))[:10]:
    print(f'  {node} (type: {data.get("type","?")})')

In [ ]:
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from graph_builder import visualize_graph

# Vẽ và lưu đồ thị tri thức
visualize_graph(G, title='Knowledge Graph - Tech Companies', save_path='knowledge_graph.png')
print('\n✅ Đồ thị đã được lưu: knowledge_graph.png')

## Phần 5: Bước 3 - GraphRAG Querying (2-hop traversal)

**Quy trình:**
1. Nhận câu hỏi từ người dùng
2. Trích xuất thực thể chính trong câu hỏi
3. Tìm node trong đồ thị và duyệt 2-hop lân cận
4. Gộp thông tin thành đoạn văn (Textualization) rồi gửi cho LLM

In [ ]:
from graph_rag import GraphRAG

grag = GraphRAG()

# Test câu hỏi đơn giản
result = grag.answer('OpenAI được thành lập năm nào?')
print(f'Query  : {result["query"]}')
print(f'Entity : {result["entity"]}')
print(f'\n=== Context từ đồ thị (2-hop) ===')
print(result['context'])
print(f'\n=== Answer ===')
print(result['answer'])
print(f'\nLatency: {result["latency_ms"]} ms')

In [ ]:
# Test câu hỏi phức tạp - yêu cầu 2-hop reasoning
result2 = grag.answer('Người sáng lập OpenAI còn liên quan đến công ty xe điện nào?')
print(f'Query  : {result2["query"]}')
print(f'Entity : {result2["entity"]}')
print(f'Nodes retrieved: {result2["nodes_retrieved"]}')
print(f'\n=== Context (2-hop) ===')
print(result2['context'][:600])
print(f'\n=== Answer ===')
print(result2['answer'])

## Phần 6: Flat RAG (TF-IDF - ChromaDB/Faiss simulation)

Hệ thống RAG truyền thống chỉ dùng similarity search, **không có khả năng suy luận đa bước**.

In [ ]:
from flat_rag import FlatRAG

flat = FlatRAG()

# Cùng câu hỏi phức tạp
result_flat = flat.answer('Người sáng lập OpenAI còn liên quan đến công ty xe điện nào?')
print(f'Query: {result_flat["query"]}')
print(f'\nTop-3 retrieved docs:')
for r in result_flat['retrieved_docs']:
    print(f'  [score={r["score"]}] {r["doc"]}')
print(f'\nAnswer: {result_flat["answer"]}')
print('\n⚠️  FlatRAG không thể kết nối: Elon Musk → OpenAI → Tesla (2-hop)')

## Phần 7: Bước 4 - Đánh giá 20 câu hỏi benchmark

So sánh Flat RAG và GraphRAG trên 20 câu hỏi từ đơn giản đến phức tạp.

In [ ]:
from evaluation import run_evaluation, save_results_csv
import pandas as pd

results, stats = run_evaluation()
save_results_csv(results)

In [ ]:
# Hiển thị bảng kết quả
df = pd.DataFrame(results)
cols = ['id','question','flat_latency_ms','graph_latency_ms','flat_hallucinates','graph_correct']
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)
print(df[cols].to_string(index=False))

In [ ]:
# Biểu đồ so sánh latency
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

df = pd.DataFrame(results)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Latency comparison
axes[0].bar(df['id'], df['flat_latency_ms'], alpha=0.7, label='FlatRAG', color='#F97B4F')
axes[0].bar(df['id'], df['graph_latency_ms'], alpha=0.7, label='GraphRAG', color='#4F8EF7')
axes[0].set_title('Latency per Query (ms)', fontweight='bold')
axes[0].set_xlabel('Query ID')
axes[0].set_ylabel('Time (ms)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Hallucination pie
hallu = sum(1 for r in results if 'YES' in str(r.get('flat_hallucinates','')))
ok = len(results) - hallu
axes[1].pie([ok, hallu], labels=['Correct', 'Hallucination'],
            colors=['#4FD6A0','#F97B4F'], autopct='%1.0f%%', startangle=90)
axes[1].set_title('FlatRAG Hallucination Rate', fontweight='bold')

plt.tight_layout()
plt.savefig('benchmark_chart.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Chart saved: benchmark_chart.png')

## Phần 8: Phân tích chi phí (Token Usage & Time)

In [ ]:
from evaluation import cost_analysis
cost_analysis(stats)

## Phần 9: Kết luận

### Trường hợp FlatRAG ảo giác - GraphRAG trả lời đúng

| # | Câu hỏi | FlatRAG | GraphRAG |
|---|---------|---------|----------|
| 1 | Người sáng lập OpenAI liên quan đến xe điện nào? | ❌ Không kết nối được | ✅ Tesla (Elon Musk 2-hop) |
| 2 | CEO Apple trước Tim Cook xây dựng sản phẩm gì? | ❌ Ảo giác | ✅ iPhone, iOS (Steve Jobs) |
| 3 | Công ty phát triển AlphaGo được ai mua? | ❌ Thiếu context | ✅ Google → DeepMind |
| 4 | Người sáng lập Google học ở đâu? | ❌ Không rõ | ✅ Đại học Stanford |
| 5 | AI model nào ra mắt 2023? | ❌ Không đầy đủ | ✅ GPT-4, Gemini |
| 6 | Elon Musk thành lập những công ty nào? | ❌ Thiếu | ✅ OpenAI, Tesla, SpaceX, X |

### Bảng so sánh tổng hợp

| Tiêu chí | Flat RAG | GraphRAG |
|---------|----------|----------|
| Độ chính xác | Trung bình | Cao |
| Suy luận đa bước | ❌ Không | ✅ Có (2-hop) |
| Chống ảo giác | ❌ Yếu | ✅ Tốt |
| Tốc độ retrieval | Nhanh | Vừa |
| Chi phí token | Cao | Thấp hơn |
| Trực quan hóa | ❌ Không | ✅ Có |

**→ GraphRAG vượt trội với câu hỏi phức tạp cần suy luận đa quan hệ.**